# Агент «Куда сходить в Алматы»

**Пайплайн:** sxodim.com → Jina Reader → GPT-5-mini (структурирование) → LlamaIndex RAG → агент

Логика живёт в `src/`, ноутбук её импортирует — так код не дублируется между `.py` и `.ipynb`.


## 0. Настройка

**В Colab** достаточно запустить ячейку ниже — она клонирует репозиторий (ноутбук импортирует из `src/`, поэтому одного `.ipynb` недостаточно), поставит зависимости и возьмёт ключ из панели Secrets (🔑 слева). Добавь туда `OPENAI_API_KEY` до запуска. GPU не нужен — хватит CPU runtime.

**Локально** ячейка ничего не делает: нужен `pip install -r requirements.txt` и `OPENAI_API_KEY` в `.env`.


In [ ]:
REPO_URL = "https://github.com/zhadyrazhan/shodim-almaty-agent.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    if not Path("shodim-almaty-agent").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if Path("shodim-almaty-agent").exists():
        os.chdir("shodim-almaty-agent")

    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab: репозиторий и зависимости готовы, ключ загружен")
else:
    print("Локальный запуск — ключ берётся из .env")

In [ ]:
import json

from src import scraper, extract, agent
from src.config import SXODIM_DATA, RAW_DIR, LLM_BACKEND

print("LLM backend:", LLM_BACKEND)

## 1. Парсинг сайта (Jina Reader)

Jina Reader отдаёт готовый markdown по любому URL, поэтому не нужно писать CSS-селекторы под вёрстку, которая может измениться.

In [ ]:
written = scraper.scrape_all()
for name, path in written.items():
    print(f"{name}: {path.stat().st_size:,} байт")

### Фрагмент спарсенных данных

In [ ]:
raw = (RAW_DIR / "afisha.md").read_text(encoding="utf-8")
print(raw[:800])

## 2. Структурирование в JSON

Спарсенный markdown шумный (меню, реклама, дубли ссылок), поэтому он режется на чанки и передаётся модели со схемой Pydantic — structured output сам приводит всё к типам.

In [ ]:
records = extract.extract_all()
print(f"\nвсего записей: {len(records)}")

### Структурированный JSON (пример)

In [ ]:
data = json.loads(SXODIM_DATA.read_text(encoding="utf-8"))
print(f"записей: {len(data)}\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

In [ ]:
from collections import Counter
print("по типам:", dict(Counter(d["kind"] for d in data)))
print("по категориям:", dict(Counter(d["category"] for d in data).most_common(10)))

## 3. Агент (LlamaIndex + OpenAI)

Записи индексируются в `VectorStoreIndex`, поверх — query engine с системным промптом гида.

In [ ]:
TEST_QUESTIONS = [
    "Куда сходить на выходных?",
    "Посоветуй место для свидания",
    "Куда сводить ребенка?",
    "Какие концерты будут?",
    "Где вкусно поесть?",
]

answers = {}
for q in TEST_QUESTIONS:
    a = agent.ask(q)
    answers[q] = a
    print(f"Q: {q}\nA: {a}\n{'-' * 70}")

## 4. Сохранение примеров диалогов

Обязательный дилеверабл `agent_examples.md` — 5+ примеров.

In [ ]:
lines = ["# Примеры диалогов с агентом\n"]
for q, a in answers.items():
    lines.append(f"\n## {q}\n\n{a}\n")

(agent.SXODIM_DATA.parent.parent / "agent_examples.md").write_text(
    "".join(lines), encoding="utf-8"
)
print(f"сохранено {len(answers)} диалогов в agent_examples.md")

## 5. Скачать результаты

Файлы лежат внутри runtime и исчезнут вместе с сессией, поэтому забери их сразу. Сам ноутбук скачивается отдельно: File → Download → Download .ipynb — **после** того, как все ячейки отработали, чтобы выводы сохранились.

In [ ]:
ARTIFACTS = ["agent_examples.md", "data/sxodim_data.json"]

if IN_COLAB:
    from google.colab import files

    for path in ARTIFACTS:
        files.download(path)
else:
    from pathlib import Path

    for path in ARTIFACTS:
        p = Path(path)
        print(f"{path}: {'есть' if p.exists() else 'НЕТ'}"
              f"{f' ({p.stat().st_size:,} байт)' if p.exists() else ''}")